# Assignment

## Brief

Write the Python codes for the following questions.

## Instructions

Paste the answer as Python in the answer code section below each question.

### Question 1

Question: Implement a simple Thrift server and client that defines a `Student` struct with fields `name` (string), `age` (integer), and `courses` (list of strings). Include a service `School` with a method `enrollCourse` that takes a `Student` record and a course name, adds the course to the student's course list, and returns the updated `Student` record.

Answer:

**Thrift schema (student.thrift)**

In [11]:
%%writefile student.thrift

struct Student {
  1: required string name,
  2: optional i32 age,
  3: optional list<string> courses
}

service School {
  Student enrollCourse(1: required Student student, 2: required string course)
}

Overwriting student.thrift


**Thrift server (student_server.py)**

In [8]:
%%writefile student_thrift_server.py
import thriftpy2
from thriftpy2.rpc import make_server

student_thrift = thriftpy2.load("student.thrift", module_name="student_thrift")

class School(object):
    def enrollCourse(self, student, course):
        if student.courses is None:
            student.courses = []
        student.courses.append(course)
        return student

server = make_server(student_thrift.School, School(), '127.0.0.1', 6000, client_timeout=None)
print("Student Thrift server started on 127.0.0.1:6000")
server.serve()

Overwriting student_thrift_server.py


Run the thrift server

**Thrift client (student_client.py)**

In [14]:
import thriftpy2
from thriftpy2.rpc import make_client

student_thrift = thriftpy2.load("student.thrift", module_name="student_thrift")
school = make_client(student_thrift.School, '127.0.0.1', 6000, timeout=None)

student = student_thrift.Student(name="Alice", age=20, courses=["math"])
print("Before enrollment:", student.courses)

student = school.enrollCourse(student, "physics")
print("After enrollment:", student.courses)

Before enrollment: ['math']
After enrollment: ['math', 'physics']


### Question 2

Question: Implement a simple Protocol Buffers server and client that defines a `Book` message with fields `title` (string), `author` (string), and `page_count` (integer). Include a service `Library` with a method `checkoutBook` that takes a `Book` message and returns the same `Book` message.

Answer:

**Protobuf schema (book.proto)**

In [15]:
%%writefile book.proto
syntax = "proto3";

package library;

message Book {
  string title = 1;
  string author = 2;
  int32 page_count = 3;
}

service Library {
  rpc checkoutBook(Book) returns (Book);
}

Overwriting book.proto


Run

**Protobuf server (book_server.py)**

In [16]:
%%writefile book_server.py
from concurrent import futures
import grpc
import book_pb2
import book_pb2_grpc

class LibraryServicer(book_pb2_grpc.LibraryServicer):
    def checkoutBook(self, request, context):
        return book_pb2.Book(
            title=request.title,
            author=request.author,
            page_count=request.page_count,
        )

def serve():
    server = grpc.server(futures.ThreadPoolExecutor(max_workers=2))
    book_pb2_grpc.add_LibraryServicer_to_server(LibraryServicer(), server)
    server.add_insecure_port('[::]:50051')
    server.start()
    print('Book gRPC server listening on 50051')
    server.wait_for_termination()

if __name__ == '__main__':
    serve()

Overwriting book_server.py


Run

**Protobuf client (book_client.py)**

In [17]:
import grpc
import book_pb2
import book_pb2_grpc

with grpc.insecure_channel('localhost:50051') as channel:
    stub = book_pb2_grpc.LibraryStub(channel)
    book = book_pb2.Book(title='1984', author='George Orwell', page_count=328)
    response = stub.checkoutBook(book)
    print('Checked out book:')
    print('  title:', response.title)
    print('  author:', response.author)
    print('  page_count:', response.page_count)

Checked out book:
  title: 1984
  author: George Orwell
  page_count: 328
